# Parameter Robustness Study for the Selected Neural Hedge

This notebook investigates whether the selected architecture remains effective when the **Black--Scholes market parameters** change.

This is **not architecture hyperparameter tuning**. The architecture is kept fixed:

\[
\delta_n = f_\theta\left(\log(S_{t_n}/K),\tau_n/T\right),
\]

with a shared Markov MLP and sigmoid output. The goal is to test whether this selected architecture remains close to the analytic benchmarks when we vary:

\[
K,\quad \sigma,\quad T,\quad N.
\]

We keep \(r=0\) because the current self-financing wealth expression is written in the zero-rate minimal-task convention:

\[
V_T = \pi + \sum_{n=0}^{N-1}\delta_n(S_{n+1}-S_n).
\]

The primary diagnostic is \(\mathrm{RMSE}_{NN}/\mathrm{RMSE}_{DT}\), where \(DT\) is the discrete-time MSE-optimal hedge. Values close to 1 mean the neural hedge is closely matching the fairest finite-grid benchmark.

In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from dataclasses import dataclass, asdict, replace
from pathlib import Path
import json
import time
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.special import ndtr
except Exception as e:
    raise ImportError("This notebook requires scipy. In Colab run: !pip install scipy") from e


@dataclass(frozen=True)
class ExperimentConfig:
    # Black--Scholes model parameters
    S0: float = 1.0
    K: float = 0.9
    T: float = 0.5
    r: float = 0.0
    sigma: float = 0.4
    N: int = 125

    # Monte Carlo sizes. These are reduced relative to the single-setting benchmark
    # because this notebook retrains the model across several parameter regimes.
    n_train: int = 50_000
    n_val: int = 15_000
    n_test: int = 50_000

    # Selected architecture: keep fixed throughout robustness study.
    width: int = 64
    depth: int = 3
    hidden_activation: str = "tanh"
    output_activation: str = "sigmoid"

    # Training
    batch_size: int = 4096
    learning_rate: float = 1e-3
    max_epochs: int = 150
    patience: int = 15

    # Reproducibility and output
    seed: int = 123
    output_dir: str = "parameter_robustness_outputs"

    # Start with quick_run=True to smoke-test in Colab.
    # Set quick_run=False for final report-grade robustness runs.
    quick_run: bool = True


base_cfg = ExperimentConfig()

if base_cfg.quick_run:
    base_cfg = replace(
        base_cfg,
        n_train=5_000,
        n_val=2_000,
        n_test=5_000,
        max_epochs=5,
        patience=3,
    )

out_dir = Path(base_cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(base_cfg.seed)

print("Base configuration:")
print(json.dumps(asdict(base_cfg), indent=2))

if abs(base_cfg.r) > 1e-12:
    print("WARNING: This notebook assumes r=0 for the current terminal wealth convention.")

In [ ]:
# ============================================================
# 2. Black--Scholes utilities and path simulation
# ============================================================

def norm_cdf(x):
    return ndtr(x)


def get_grids(cfg):
    dt = cfg.T / cfg.N
    t_grid = np.arange(cfg.N, dtype=np.float64) * dt
    tau_grid = cfg.T - t_grid
    tau_norm_grid = tau_grid / cfg.T
    return dt, t_grid, tau_grid, tau_norm_grid


def bs_call_price(S, tau, cfg):
    S = np.asarray(S, dtype=np.float64)
    tau = np.asarray(tau, dtype=np.float64)
    intrinsic = np.maximum(S - cfg.K, 0.0)
    tau_safe = np.maximum(tau, 1e-12)
    d1 = (np.log(S / cfg.K) + (cfg.r + 0.5 * cfg.sigma**2) * tau_safe) / (cfg.sigma * np.sqrt(tau_safe))
    d2 = d1 - cfg.sigma * np.sqrt(tau_safe)
    price = S * norm_cdf(d1) - cfg.K * np.exp(-cfg.r * tau_safe) * norm_cdf(d2)
    return np.where(tau <= 0, intrinsic, price)


def bs_call_delta(S, tau, cfg):
    S = np.asarray(S, dtype=np.float64)
    tau = np.asarray(tau, dtype=np.float64)
    tau_safe = np.maximum(tau, 1e-12)
    d1 = (np.log(S / cfg.K) + (cfg.r + 0.5 * cfg.sigma**2) * tau_safe) / (cfg.sigma * np.sqrt(tau_safe))
    delta = norm_cdf(d1)
    expiry_delta = (S > cfg.K).astype(np.float64)
    return np.where(tau <= 0, expiry_delta, delta)


def simulate_gbm_paths(n_paths, cfg, seed=None):
    rng = np.random.default_rng(seed)
    dt = cfg.T / cfg.N
    Z = rng.standard_normal(size=(n_paths, cfg.N)).astype(np.float32)
    increments = (cfg.r - 0.5 * cfg.sigma**2) * dt + cfg.sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(increments, axis=1)
    paths = np.empty((n_paths, cfg.N + 1), dtype=np.float32)
    paths[:, 0] = cfg.S0
    paths[:, 1:] = cfg.S0 * np.exp(log_paths)
    return paths


def call_payoff(S_T, cfg):
    return np.maximum(S_T - cfg.K, 0.0)

In [ ]:
# ============================================================
# 3. Hedge-error and benchmark utilities
# ============================================================

def hedge_error(paths, deltas, premium, cfg):
    dS = paths[:, 1:] - paths[:, :-1]
    terminal_wealth = premium + np.sum(deltas * dS, axis=1)
    payoff = call_payoff(paths[:, -1], cfg)
    return terminal_wealth - payoff


def cvar_loss_from_he(he, alpha=0.95):
    loss = -np.asarray(he)
    var_alpha = np.quantile(loss, alpha)
    tail = loss[loss >= var_alpha]
    return float(np.mean(tail))


def strategy_metrics(strategy, he, premium, cfg, experiment_name, varied_param, varied_value, runtime_s=None, extra=None):
    he = np.asarray(he)
    loss = -he
    bs_price = float(bs_call_price(cfg.S0, cfg.T, cfg))
    row = {
        "experiment": experiment_name,
        "varied_param": varied_param,
        "varied_value": varied_value,
        "strategy": strategy,
        "S0": cfg.S0,
        "K": cfg.K,
        "T": cfg.T,
        "r": cfg.r,
        "sigma": cfg.sigma,
        "N": cfg.N,
        "premium": float(premium),
        "bs_price": bs_price,
        "premium_error": float(premium - bs_price),
        "mean_he": float(np.mean(he)),
        "std_he": float(np.std(he, ddof=1)),
        "rmse": float(np.sqrt(np.mean(he**2))),
        "he_q01": float(np.quantile(he, 0.01)),
        "he_q05": float(np.quantile(he, 0.05)),
        "loss_var95": float(np.quantile(loss, 0.95)),
        "loss_cvar95": cvar_loss_from_he(he, 0.95),
        "loss_var99": float(np.quantile(loss, 0.99)),
        "loss_cvar99": cvar_loss_from_he(he, 0.99),
    }
    if runtime_s is not None:
        row["runtime_s"] = float(runtime_s)
    if extra:
        row.update(extra)
    return row


def bs_delta_hedge(paths, cfg):
    _, _, tau_grid, _ = get_grids(cfg)
    S = paths[:, :-1].astype(np.float64)
    tau = tau_grid[None, :]
    return bs_call_delta(S, tau, cfg).astype(np.float32)


def dt_mse_optimal_delta_hedge(paths, cfg):
    """
    Discrete-time MSE-optimal hedge for the r=0 Black--Scholes minimal task.
    """
    if abs(cfg.r) > 1e-12:
        raise ValueError("This closed-form DT benchmark is implemented only for r=0.")
    h, _, tau_grid, _ = get_grids(cfg)
    S = paths[:, :-1].astype(np.float64)
    tau = tau_grid[None, :].astype(np.float64)
    tau_safe = np.maximum(tau, 1e-12)
    sqrt_tau = np.sqrt(tau_safe)
    d1 = (np.log(S / cfg.K) + 0.5 * cfg.sigma**2 * tau_safe) / (cfg.sigma * sqrt_tau)
    d2 = d1 - cfg.sigma * sqrt_tau
    C = bs_call_price(S, tau, cfg)
    shift = cfg.sigma * h / sqrt_tau
    numerator = (
        S**2 * np.exp(cfg.sigma**2 * h) * norm_cdf(d1 + shift)
        - cfg.K * S * norm_cdf(d2 + shift)
        - S * C
    )
    denominator = S**2 * (np.exp(cfg.sigma**2 * h) - 1.0)
    delta = numerator / denominator
    return delta.astype(np.float32)

In [ ]:
# ============================================================
# 4. TensorFlow neural hedge model
# ============================================================

try:
    import tensorflow as tf
except Exception as e:
    raise ImportError(
        "TensorFlow is required. In Colab, use Runtime > Change runtime type > GPU. TensorFlow should already be installed."
    ) from e


class SharedMarkovHedge(tf.keras.Model):
    def __init__(self, cfg, initial_premium):
        super().__init__()
        self.cfg = cfg
        _, _, _, tau_norm_grid = get_grids(cfg)
        self.premium = self.add_weight(
            name="premium",
            shape=(),
            initializer=tf.keras.initializers.Constant(initial_premium),
            trainable=True,
            dtype=tf.float32,
        )
        layers = [tf.keras.layers.InputLayer(input_shape=(2,))]
        for _ in range(cfg.depth):
            layers.append(tf.keras.layers.Dense(cfg.width, activation=cfg.hidden_activation))
        layers.append(tf.keras.layers.Dense(1, activation=cfg.output_activation))
        self.net = tf.keras.Sequential(layers)
        self.tau_norm_tf = tf.constant(tau_norm_grid.astype(np.float32), dtype=tf.float32)

    def make_features(self, paths):
        S = paths[:, :-1]
        log_m = tf.math.log(S / tf.constant(self.cfg.K, dtype=tf.float32))
        tau_feat = tf.broadcast_to(self.tau_norm_tf[None, :], tf.shape(S))
        return tf.stack([log_m, tau_feat], axis=-1)

    def call(self, paths, training=False):
        features = self.make_features(paths)
        batch_size = tf.shape(paths)[0]
        flat_features = tf.reshape(features, (-1, 2))
        flat_delta = self.net(flat_features, training=training)
        deltas = tf.reshape(flat_delta[:, 0], (batch_size, self.cfg.N))
        dS = paths[:, 1:] - paths[:, :-1]
        payoff = tf.nn.relu(paths[:, -1] - tf.constant(self.cfg.K, dtype=tf.float32))
        terminal_wealth = self.premium + tf.reduce_sum(deltas * dS, axis=1)
        he = terminal_wealth - payoff
        return he, deltas


def train_neural_hedge(paths_train, paths_val, cfg, seed_offset=0):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(cfg.seed + seed_offset)
    initial_premium = float(bs_call_price(cfg.S0, cfg.T, cfg))
    model = SharedMarkovHedge(cfg, initial_premium)
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
    train_ds = (
        tf.data.Dataset.from_tensor_slices(paths_train.astype(np.float32))
        .shuffle(buffer_size=min(len(paths_train), 50_000), seed=cfg.seed + seed_offset, reshuffle_each_iteration=True)
        .batch(cfg.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices(paths_val.astype(np.float32))
        .batch(cfg.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            he, _ = model(batch, training=True)
            loss = tf.reduce_mean(tf.square(he))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function
    def eval_step(batch):
        he, _ = model(batch, training=False)
        return tf.reduce_sum(tf.square(he)), tf.shape(he)[0]

    best_val = np.inf
    best_weights = None
    wait = 0
    history = []
    start_time = time.time()
    for epoch in range(1, cfg.max_epochs + 1):
        train_losses = []
        for batch in train_ds:
            loss = train_step(batch)
            train_losses.append(float(loss.numpy()))
        val_sse = 0.0
        val_n = 0
        for batch in val_ds:
            sse, n = eval_step(batch)
            val_sse += float(sse.numpy())
            val_n += int(n.numpy())
        train_loss = float(np.mean(train_losses))
        val_loss = val_sse / val_n
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "premium": float(model.premium.numpy()),
            "elapsed_s": time.time() - start_time,
        })
        if val_loss < best_val - 1e-10:
            best_val = val_loss
            best_weights = model.get_weights()
            wait = 0
        else:
            wait += 1
        if epoch == 1 or epoch % 10 == 0 or wait == 0:
            print(f"  epoch {epoch:03d} | train {train_loss:.8e} | val {val_loss:.8e} | premium {float(model.premium.numpy()):.8f} | wait {wait}")
        if wait >= cfg.patience:
            print(f"  early stopping at epoch {epoch}; best val loss = {best_val:.8e}")
            break
    if best_weights is not None:
        model.set_weights(best_weights)
    return model, pd.DataFrame(history)


def predict_neural_hedge(model, paths, cfg):
    ds = tf.data.Dataset.from_tensor_slices(paths.astype(np.float32)).batch(cfg.batch_size)
    all_he = []
    all_deltas = []
    for batch in ds:
        he, deltas = model(batch, training=False)
        all_he.append(he.numpy())
        all_deltas.append(deltas.numpy())
    return np.concatenate(all_he), np.concatenate(all_deltas, axis=0)

In [ ]:
# ============================================================
# 5. Define one-factor-at-a-time robustness grid
# ============================================================

def make_experiment_grid(base_cfg):
    experiments = []
    def add(name, varied_param, varied_value, cfg):
        experiments.append({"experiment": name, "varied_param": varied_param, "varied_value": varied_value, "cfg": cfg})
    add("baseline", "baseline", "baseline", base_cfg)
    for K in [0.8, 0.9, 1.0, 1.1, 1.2]:
        if K != base_cfg.K:
            add(f"K={K}", "K", K, replace(base_cfg, K=K))
    for sigma in [0.2, 0.4, 0.6]:
        if sigma != base_cfg.sigma:
            add(f"sigma={sigma}", "sigma", sigma, replace(base_cfg, sigma=sigma))
    for T in [0.25, 0.5, 1.0]:
        if T != base_cfg.T:
            add(f"T={T}", "T", T, replace(base_cfg, T=T))
    for N in [30, 60, 125, 250]:
        if N != base_cfg.N:
            add(f"N={N}", "N", N, replace(base_cfg, N=N))
    return experiments

experiments = make_experiment_grid(base_cfg)
print(f"Number of robustness settings: {len(experiments)}")
for i, exp in enumerate(experiments, 1):
    cfg = exp["cfg"]
    print(f"{i:02d}. {exp['experiment']:12s} | K={cfg.K}, sigma={cfg.sigma}, T={cfg.T}, N={cfg.N}")

In [ ]:
# ============================================================
# 6. Run a single robustness setting
# ============================================================

def stable_seed_from_name(name):
    # Avoid Python's randomized hash so results are reproducible across sessions.
    return sum((i + 1) * ord(ch) for i, ch in enumerate(name)) % 100_000


def run_one_setting(exp, base_seed=123):
    cfg = exp["cfg"]
    experiment_name = exp["experiment"]
    varied_param = exp["varied_param"]
    varied_value = exp["varied_value"]
    print("\n" + "=" * 80)
    print(f"Running setting: {experiment_name}")
    print(json.dumps(asdict(cfg), indent=2))
    print("=" * 80)
    start_total = time.time()
    seed_shift = stable_seed_from_name(experiment_name)
    paths_train = simulate_gbm_paths(cfg.n_train, cfg, seed=base_seed + seed_shift + 1)
    paths_val = simulate_gbm_paths(cfg.n_val, cfg, seed=base_seed + seed_shift + 2)
    paths_test = simulate_gbm_paths(cfg.n_test, cfg, seed=base_seed + seed_shift + 3)
    bs_price_0 = float(bs_call_price(cfg.S0, cfg.T, cfg))
    rows = []

    t0 = time.time()
    deltas_no = np.zeros((cfg.n_test, cfg.N), dtype=np.float32)
    he_no = hedge_error(paths_test, deltas_no, bs_price_0, cfg)
    rows.append(strategy_metrics("No hedge", he_no, bs_price_0, cfg, experiment_name, varied_param, varied_value, runtime_s=time.time() - t0))

    t0 = time.time()
    deltas_bs = bs_delta_hedge(paths_test, cfg)
    he_bs = hedge_error(paths_test, deltas_bs, bs_price_0, cfg)
    rows.append(strategy_metrics("Black--Scholes delta", he_bs, bs_price_0, cfg, experiment_name, varied_param, varied_value, runtime_s=time.time() - t0))

    t0 = time.time()
    deltas_dt = dt_mse_optimal_delta_hedge(paths_test, cfg)
    he_dt = hedge_error(paths_test, deltas_dt, bs_price_0, cfg)
    rows.append(strategy_metrics("Discrete-time MSE-optimal", he_dt, bs_price_0, cfg, experiment_name, varied_param, varied_value, runtime_s=time.time() - t0))

    t0 = time.time()
    model, history_df = train_neural_hedge(paths_train, paths_val, cfg, seed_offset=seed_shift)
    train_runtime = time.time() - t0
    he_nn, _ = predict_neural_hedge(model, paths_test, cfg)
    nn_premium = float(model.premium.numpy())
    rows.append(strategy_metrics(
        "Neural hedge", he_nn, nn_premium, cfg, experiment_name, varied_param, varied_value,
        runtime_s=train_runtime,
        extra={"epochs": int(history_df["epoch"].max()), "best_val_loss": float(history_df["val_loss"].min())},
    ))

    safe_name = experiment_name.replace("=", "_").replace(".", "p")
    history_df.to_csv(out_dir / f"history_{safe_name}.csv", index=False)
    total_runtime = time.time() - start_total
    for row in rows:
        row["total_setting_runtime_s"] = total_runtime
    print(f"Completed {experiment_name} in {total_runtime:.1f}s")

    del paths_train, paths_val, paths_test, deltas_no, deltas_bs, deltas_dt
    del he_no, he_bs, he_dt, he_nn, model
    gc.collect()
    tf.keras.backend.clear_session()
    return rows

In [ ]:
# ============================================================
# 7. Run robustness study with resume support
# ============================================================

summary_path = out_dir / "parameter_robustness_summary.csv"

if summary_path.exists():
    existing = pd.read_csv(summary_path)
    completed_experiments = set(existing["experiment"].unique())
    all_rows = existing.to_dict("records")
    print(f"Loaded existing results for {len(completed_experiments)} settings from {summary_path}.")
else:
    completed_experiments = set()
    all_rows = []

for exp in experiments:
    name = exp["experiment"]
    if name in completed_experiments:
        print(f"Skipping already completed setting: {name}")
        continue
    rows = run_one_setting(exp, base_seed=base_cfg.seed)
    all_rows.extend(rows)
    pd.DataFrame(all_rows).to_csv(summary_path, index=False)
    print(f"Saved progress to {summary_path}")

results_df = pd.DataFrame(all_rows)
results_df

In [ ]:
# ============================================================
# 8. Summarise NN robustness relative to analytic benchmarks
# ============================================================

results_df = pd.read_csv(summary_path)
pivot = results_df.pivot_table(
    index=["experiment", "varied_param", "varied_value", "S0", "K", "T", "r", "sigma", "N"],
    columns="strategy",
    values=["rmse", "premium", "loss_cvar95", "he_q01"],
    aggfunc="first",
)
pivot.columns = [f"{metric}_{strategy}".replace(" ", "_").replace("--", "-") for metric, strategy in pivot.columns]
pivot = pivot.reset_index()
pivot["nn_rmse_over_dt_rmse"] = pivot["rmse_Neural_hedge"] / pivot["rmse_Discrete-time_MSE-optimal"]
pivot["nn_rmse_over_bs_rmse"] = pivot["rmse_Neural_hedge"] / pivot["rmse_Black-Scholes_delta"]
pivot["nn_premium_error"] = pivot["premium_Neural_hedge"] - pivot["premium_Black-Scholes_delta"]

pivot_path = out_dir / "parameter_robustness_nn_vs_benchmarks.csv"
pivot.to_csv(pivot_path, index=False)

display_cols = [
    "experiment", "varied_param", "varied_value", "K", "sigma", "T", "N",
    "rmse_Black-Scholes_delta", "rmse_Discrete-time_MSE-optimal", "rmse_Neural_hedge",
    "nn_rmse_over_dt_rmse", "nn_premium_error", "loss_cvar95_Neural_hedge",
]
pivot[display_cols].sort_values(["varied_param", "varied_value"]).reset_index(drop=True)

In [ ]:
# ============================================================
# 9. Plot robustness results
# ============================================================

plot_df = pivot.copy()
plot_df["varied_value_numeric"] = pd.to_numeric(plot_df["varied_value"], errors="coerce")

for param in ["K", "sigma", "T", "N"]:
    sub = plot_df[plot_df["varied_param"].isin([param, "baseline"])].copy()
    if param == "K":
        sub.loc[sub["varied_param"] == "baseline", "varied_value_numeric"] = base_cfg.K
    elif param == "sigma":
        sub.loc[sub["varied_param"] == "baseline", "varied_value_numeric"] = base_cfg.sigma
    elif param == "T":
        sub.loc[sub["varied_param"] == "baseline", "varied_value_numeric"] = base_cfg.T
    elif param == "N":
        sub.loc[sub["varied_param"] == "baseline", "varied_value_numeric"] = base_cfg.N
    sub = sub.sort_values("varied_value_numeric")

    plt.figure(figsize=(8, 5))
    plt.plot(sub["varied_value_numeric"], sub["rmse_Black-Scholes_delta"], marker="o", label="Black--Scholes delta")
    plt.plot(sub["varied_value_numeric"], sub["rmse_Discrete-time_MSE-optimal"], marker="o", label="Discrete-time MSE-optimal")
    plt.plot(sub["varied_value_numeric"], sub["rmse_Neural_hedge"], marker="o", label="Neural hedge")
    plt.xlabel(param)
    plt.ylabel("Hedge-error RMSE")
    plt.title(f"Robustness to {param}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"robustness_rmse_vs_{param}.png", dpi=200)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(sub["varied_value_numeric"], sub["nn_rmse_over_dt_rmse"], marker="o")
    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.xlabel(param)
    plt.ylabel("NN RMSE / DT-optimal RMSE")
    plt.title(f"Relative NN performance vs DT-optimal: {param}")
    plt.tight_layout()
    plt.savefig(out_dir / f"robustness_ratio_vs_{param}.png", dpi=200)
    plt.show()

print(f"Saved robustness outputs to: {out_dir.resolve()}")

In [ ]:
# ============================================================
# 10. Report-ready LaTeX table
# ============================================================

latex_cols = [
    "experiment",
    "rmse_Black-Scholes_delta",
    "rmse_Discrete-time_MSE-optimal",
    "rmse_Neural_hedge",
    "nn_rmse_over_dt_rmse",
    "nn_premium_error",
]
latex_table = pivot[latex_cols].copy().sort_values("experiment")
latex_path = out_dir / "parameter_robustness_table.tex"
latex_table.to_latex(
    latex_path,
    index=False,
    float_format="%.6g",
    escape=False,
    caption="Parameter robustness of the selected neural hedge.",
    label="tab:parameter_robustness",
)
latex_table

## Suggested report interpretation

Use this experiment as a **robustness study**, not as a new hyperparameter tune.

Suggested wording:

> After selecting the shared normalised Markov MLP under the baseline minimal-task setup, we tested whether the same architecture remains effective under changes in moneyness, volatility, maturity and rebalancing frequency. For each parameter setting, the architecture was retrained from scratch but its design was kept fixed. Performance was assessed relative to the Black--Scholes delta and the discrete-time MSE-optimal hedge.

The most important column is:

\[
\frac{\mathrm{RMSE}_{NN}}{\mathrm{RMSE}_{DT}}.
\]

Values close to 1 mean the selected architecture continues to recover the finite-grid optimal hedge.